In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 359.51it/s]


In [5]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
def chunk_text(text, chunk_size, overlap):
    """
    Splits the input text into chunks of specified size with a given overlap.

    Args:
        text (str): The input text to be chunked.
        chunk_size (int): The maximum size of each chunk.
        overlap (int): The number of overlapping characters between consecutive chunks.

    Returns:
        list: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


In [7]:
text = """Ein Mathematikstudium beginnt normalerweise mit den beiden Grundvorlesungen Analysis und Lineare Algebra, wobei die lineare Algebra ein Teilgebiet der Algebra ist. Analysis und Algebra bilden die beiden Grundpfeiler der modernen Mathematik, die ihrerseits auf der Mengenlehre fußen. Wir werden uns also eine ganze Weile mit diesen beiden Bereichen beschäftigen. Soweit, so gut – doch worum geht es in beiden Vorlesungen überhaupt? Was tut ein Algebraiker und was ein Analytiker? Mit welchen Fragen beschäftigen sie sich? Es ist sinnvoll, zu Beginn einen kleinen Vorgeschmack zu geben, was uns in welchem Bereich erwartet, um den „roten Faden“ einer jeden Vorlesung besser sehen zu können.

Die Algebra, oder viel mehr eine Algebra, ist, ähnlich wie die rationalen oder die reellen Zahlen, eine Art „Zahlenraum“: In einer Algebra kann man addieren und multiplizieren. So beschäftigt sich die Algebra zu einem großen Teil mit Umformungen, die durch Addition oder Multiplikation entstehen, wobei auch das Wurzelziehen dazu gehört. In der Algebra möchte man unter anderem die Frage beantworten, wie man eine Gleichung umformen muss, um sie zu „lösen“ und ob eine bestimmte Gleichung überhaupt lösbar ist. Generell beschäftigt sich die Algebra häufig mit Gleichungen und eher selten mit Ungleichungen.

In der linearen Algebra treten alle beteiligten Größen einer Gleichung nur in erster Potenz (also in linearer Ordnung) auf. 

Eine weitere Frage der Analysis ist die, ob es Funktionen gibt, die unstetig sind, aber nirgends einen „Sprung“ machen – die Antwort hierauf lautet „ja“. Ferner können wir uns fragen, ob eine Funktion, die differenzierbar ist, eine unstetige Ableitung haben kann – auch hier ist die Antwort „ja“ – oder ob ihre Ableitung einen der eben erwähnten Sprünge aufweisen kann – diesmal lautet die Antwort „nein“. Wir folgern somit, dass eine Funktion mit Sprung niemals die Ableitung einer anderen Funktion sein kann. Sie kann aber trotzdem integrierbar sein. Die Berechnung ihres Integrals ist ebenfalls Aufgabe des Analytikers.

Die Analysis gibt uns Konzepte an die Hand, mit denen wir die aktuelle Änderung einer Funktion beschreiben können. Diese Konzepte werden in den Naturwissenschaften verwendet, um Naturgesetze oder Formeln für bestimmte Modelle aufzustellen. Dies ist auch der Grund, warum die Analysis eine so wichtige Rolle für die Naturwissenschaften spielt. Der Analytiker untersucht, wie die Veränderung eines Systems vorhergesagt werden kann, und wie exakt solche Vorhersagen sind.

Wir haben damit bereits einen kleinen Einblick in die Welt der Mathematik gewonnen und wissen zumindest grob, was uns auf welcher Seite (Algebra und Analysis) erwartet. Oft bevorzugt ein Mathematiker eine der beiden Seiten, ist also eher Algebraiker oder eher Analytiker. In Wirklichkeit kommt jedes der beiden Gebiete nicht ohne das jeweils andere aus. Es muss also nicht beim entweder – oder bleiben, denn beide Felder können gleichermaßen interessant und bereichernd sein. Zumindest in diesem Projekt werden wir uns der Analysis widmen…"""
chunks = chunk_text(text, 100, 20)
chunk_vecs = model.encode(chunks)
print(len(chunks), chunk_vecs.shape)

39 (39, 384)


In [8]:
def search(question, k=3):
    q_vec = model.encode(question)

    scores = []
    for v in chunk_vecs:
        scores.append(cosine_sim(q_vec, v))

    best = np.argsort(scores)[::-1][:k]

    return [(scores[i], chunks[i]) for i in best]

In [9]:
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

def answer(question, k=3):
    results = search(question, k)
    context = "\n\n".join(chunk for score, chunk in results)

    prompt = f"""Du bist ein Assistent, der Fragen zu Vorlesungsnotizen beantwortet.
Beantworte die Frage ausschließlich anhand des folgenden Kontexts.
Wenn die Antwort im Kontext nicht enthalten ist, antworte genau: "Das steht nicht in den Notizen."
Verwende kein eigenes Wissen und erfinde nichts.
Antworte in der Sprache der Frage.

Kontext:
{context}

Frage: {question}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    return response.text

In [11]:
import warnings
warnings.filterwarnings("ignore")
print(answer("Was kann man in Algebra machen?"))
print(answer("Was ist der Eigenwert einer Matrix?"))

In einer Algebra kann man addieren und multiplizieren.
Das steht nicht in den Notizen.
